In [ ]:
import ipywidgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from mito import ARM_Cito, Mitochondria
from n_mito.loop_simulator import LoopSimulator

tqdm.pandas()

In [ ]:
loop_sim = LoopSimulator(
    ARM_Cito,
    Mitochondria(
        CytoC_C=ARM_Cito.CytoC_C,
        Smac_C=ARM_Cito.Smac_C,
        Bax_A=ARM_Cito.Bax_A,
    ),
)


def simulate(mito_volume_fraction: list[float] | np.ndarray, /, dt: float = 1.0):
    mito_volume_fraction = np.array(mito_volume_fraction, float)
    mito_volume_fraction /= mito_volume_fraction.sum()

    t = np.linspace(0, 30_000, int(30_000 / dt))
    df = loop_sim.solve(
        main_values={ARM_Cito.L: 1000},
        loop_values={Mitochondria.volume: 0.07 * mito_volume_fraction},
        save_at=t,
        loop_output="index_as_suffix",
    )

    df.index /= 60
    df.index.rename("time [min]", inplace=True)

    columns = [
        ARM_Cito.C3_A.variable,
        ARM_Cito.C8_A.variable,
        ARM_Cito.Apop.variable,
        *df.filter(like="Mito_A_"),
    ]
    return df[columns]


def max_activity(df: pd.DataFrame):
    return df.aggregate(lambda x: (x > x.median()).idxmax())
    return df.diff().idxmax()

## Binary

In [ ]:
@ipywidgets.interact(
    p=ipywidgets.FloatLogSlider(min=-20, max=-1, base=2),
    dt=ipywidgets.FloatLogSlider(min=-1, max=2),
)
def _(p, dt):
    fraction = [p, 1 - p]
    df = simulate(fraction, dt=dt)
    _, axes = plt.subplots(2, sharex=True)
    df.plot(ax=axes[0])
    df.diff().plot(ax=axes[1])

In [ ]:
def max_activity_mito_2(p: float, /):
    fraction = [p, 1 - p]
    return simulate(fraction).pipe(max_activity).rename(p)


fractions = np.logspace(-5, -1, 30, base=2)
df_act = pd.concat(map(max_activity_mito_2, fractions), axis=1).T
df_act.plot(logx=True, marker=".", ylabel="time [min]")
df_act.head(2)

## Ternary

In [ ]:
import numpy as np
import ternary
from matplotlib.colors import CenteredNorm, Normalize

In [ ]:
@ipywidgets.interact(
    P=ipywidgets.FloatLogSlider(min=-20, max=-1, base=2),
    p=ipywidgets.FloatLogSlider(min=-20, max=-1, base=2),
)
def _(P, p):
    fraction = [(1 - P), P * p, P * (1 - p)]
    df = simulate(fraction)
    _, axes = plt.subplots(2, sharex=True)
    df.plot(ax=axes[0])
    df.diff().plot(ax=axes[1])

In [ ]:
def plot(data: pd.DataFrame, *, fig=None, norm=None):
    scale = sum(data.index[0])

    if fig is None:
        fig = plt.figure(figsize=(12, 6))

    if norm is None:
        norm = Normalize()

    axes = fig.subplots(2, 3, sharex=True, sharey=True)
    for ax, (_, col) in zip(axes.flat, data.items()):
        plot_col(col, ax=ax, norm=norm, scale=scale)


def plot_col(col: pd.Series, *, ax=None, norm=None, scale=None):
    if scale is None:
        scale = sum(col.index[0])
    if norm is None:
        norm = Normalize()

    norm.autoscale(col)
    _, ax = ternary.figure(ax, scale=scale)
    ax.heatmap(col.to_dict(), cmap="bwr", vmin=norm.vmin, vmax=norm.vmax)
    plot_style(ax)
    _ax = ax.get_axes()
    _ax.text(0, 1, col.name, transform=_ax.transAxes)
    return ax


def plot_style(tax):
    tax.boundary(linewidth=1)
    tax.ticks(
        ticks=[f"{x:.2e}" for x in 2.0 ** -np.arange(0, 33 + 1, 11)],
        axis="lbr",
        multiple=11,
        offset=0.05,
    )
    tax.gridlines(color="black", multiple=5)
    tax.gridlines(color="gray", multiple=1, linewidth=0.5)
    tax.clear_matplotlib_ticks()
    tax.get_axes().spines.clear()

In [ ]:
data = pd.Series(ternary.helpers.simplex_iterator(3 * 11))
data.index = data
data: pd.DataFrame = (
    data.map(lambda x: 2.0 ** -np.array(x, float))
    .progress_map(lambda x: simulate(x).pipe(max_activity))
    .apply(pd.Series)
)

In [ ]:
plot(data)

In [ ]:
plot(
    data - data[data.index.map(set).map(len) == 1].iloc[0],
    norm=CenteredNorm(),
    fig=plt.figure(figsize=(20, 10)),
)